In [0]:
from pyspark.sql import SparkSession
from datetime import date, timedelta
import random
import uuid

spark = SparkSession.builder.getOrCreate()

random.seed(42)

In [0]:
# config

CATALOG_NAME = "workspace"
SCHEMA_NAME = "metadata"
VOLUME_NAME = "raw_files"

BASE_VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"
RAW_PATH = f"{BASE_VOLUME_PATH}/synthetic_ecommerce"

NUM_CUSTOMERS = 1000
NUM_PRODUCTS = 300
NUM_ORDERS = 3000

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

In [0]:
# helpers

first_names = [
    'John', 'Jane', 'Michael', 'Sarah', 'David', 'Emily', 'Robert', 'Jessica', 'William', 'Ashley', 'James', 'Amanda', 'Christopher', 'Jennifer', 'Daniel', 'Lisa', 'Matthew', 'Nancy', 'Anthony', 'Karen', 'Mark', 'Betty', 'Donald', 'Helen', 'Steven', 'Sandra', 'Paul', 'Donna', 'Andrew', 'Carol', 'Joshua', 'Ruth', 'Kenneth', 'Sharon', 'Kevin', 'Michelle', 'Brian', 'Laura', 'George', 'Timothy', 'Kimberly', 'Ronald', 'Deborah', 'Jason', 'Dorothy', 'Edward', 'Jeffrey', 'Alex', 'Maria', 'Carlos', 'Anna', 'Thomas', 'Patricia', 'Charles', 'Linda', 'Joseph', 'Barbara', 'Richard', 'Elizabeth', 'Susan', 'Ryan', 'Jacob', 'Gary', 'Nicholas', 'Eric', 'Jonathan', 'Amy', 'Stephen', 'Angela', 'Larry', 'Justin', 'Brenda', 'Scott', 'Emma', 'Brandon', 'Olivia', 'Benjamin', 'Cynthia', 'Samuel', 'Marie', 'Gregory', 'Janet', 'Alexander', 'Catherine', 'Patrick', 'Frances', 'Jack', 'Christine', 'Dennis', 'Samantha', 'Jerry', 'Debra', 'Tyler', 'Rachel', 'Aaron', 'Carolyn', 'Jose', 'Henry', 'Virginia', 'Adam', 'Douglas', 'Heather', 'Nathan', 'Diane', 'Peter', 'Julie', 'Zachary', 'Joyce', 'Kyle', 'Victoria', 'Noah', 'Kelly', 'Alan', 'Christina', 'Ethan', 'Joan', 'Jeremy', 'Evelyn', 'Mason', 'Judith', 'Christian', 'Megan', 'Austin', 'Cheryl', 'Jesse', 'Andrea', 'Jordan', 'Hannah', 'Bryan', 'Jacqueline', 'Sean', 'Martha', 'Connor', 'Gloria', 'Caleb', 'Teresa', 'Evan', 'Sara', 'Ian', 'Janice', 'Julia', 'Adrian', 'Hunter', 'Madison', 'Cameron', 'Grace', 'Blake', 'Judy', 'Luke', 'Theresa', 'Beverly', 'Gavin', 'Denise', 'Isaac', 'Marilyn', 'Amber', 'Wyatt', 'Danielle', 'Jayden', 'Rose', 'Grayson', 'Brittany', 'Leo', 'Diana', 'Owen', 'Abigail', 'Gabriel', 'Julian', 'Lori', 'Aiden', 'Kathryn', 'Lincoln', 'Stephanie', 'Lucas', 'Nicole', 'Sebastian', 'Jackson'
]

last_names = [
    'Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez', 
    'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 
    'Lee', 'Perez', 'Thompson', 'White', 'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson',  'Walker', 'Young', 'Allen', 'King', 'Wright', 'Scott', 'Torres', 'Nguyen', 'Hill', 'Flores', 'Green', 'Adams', 'Nelson', 'Baker', 'Hall', 'Rivera', 'Campbell', 'Mitchell', 'Carter', 'Roberts', 'Gomez', 'Phillips', 'Evans', 'Turner', 'Diaz', 'Parker', 'Cruz', 'Edwards', 'Collins', 'Reyes', 'Stewart', 'Morris', 'Morales', 'Murphy', 'Cook', 'Rogers', 'Gutierrez', 'Ortiz', 'Morgan', 'Cooper', 'Peterson', 'Bailey', 'Reed', 'Kelly', 'Howard', 'Ramos', 'Kim', 'Cox', 'Ward', 'Richardson', 'Watson', 'Brooks', 'Chavez', 'Wood', 'James', 'Bennett', 'Gray', 'Mendoza', 'Ruiz', 'Hughes', 'Price', 'Alvarez', 'Castillo', 'Sanders', 'Patel', 'Myers', 'Long', 'Ross', 'Foster', 'Jimenez', 'Powell', 'Jenkins', 'Perry', 'Russell', 'Sullivan', 'Bell', 'Coleman', 'Butler', 'Henderson', 'Barnes', 'Gonzales', 'Fisher', 'Vasquez', 'Simmons', 'Romero', 'Jordan', 'Patterson', 'Alexander', 'Hamilton', 'Graham', 'Reynolds', 'Griffin', 'Wallace', 'Moreno', 'West', 'Cole', 'Hayes', 'Bryant', 'Herrera', 'Gibson', 'Ellis', 'Tran', 'Medina', 'Aguilar', 'Stevens', 'Murray', 'Ford', 'Castro', 'Marshall', 'Owen', 'Harrison', 'Fernandez', 'McDonald', 'Woods', 'Washington', 'Kennedy', 'Wells', 'Vargas', 'Henry', 'Chen', 'Freeman', 'Webb', 'Tucker', 'Guzman', 'Burns', 'Crawford', 'Olson', 'Simpson', 'Porter', 'Hunter', 'Gordon', 'Mendez', 'Silva', 'Shaw', 'Snyder', 'Mason', 'Dixon', 'Munoz', 'Hunt', 'Hicks', 'Holmes', 'Palmer', 'Wagner', 'Black', 'Robertson', 'Boyd', 'Rose', 'Stone', 'Salazar', 'Fox', 'Warren', 'Mills', 'Meyer', 'Rice', 'Schmidt', 'Garza', 'Daniels', 'Ferguson', 'Nichols', 'Stephens', 'Soto', 'Weaver', 'Ryan', 'Gardner', 'Payne', 'Grant', 'Dunn', 'Kelley', 'Spencer', 'Hawkins', 'Arnold', 'Pierce', 'Vazquez', 'Hansen', 'Peters', 'Santos', 'Hart', 'Bradley', 'Knight', 'Elliott', 'Cunningham', 'Duncan', 'Armstrong', 'Hudson', 'Carroll', 'Lane', 'Riley', 'Andrews', 'Alvarado', 'Ray', 'Delgado', 'Berry', 'Perkins', 'Hoffman', 'Johnston', 'Matthews', 'Pena', 'Richards', 'Contreras', 'Willis', 'Carpenter', 'Lawrence', 'Sandoval', 'Guerrero', 'George', 'Chapman', 'Rios', 'Estrada', 'Ortega', 'Watkins', 'Greene', 'Nunez', 'Wheeler', 'Valdez', 'Harper', 'Burke', 'Larson', 'Santana', 'Mann', 'Zimmerman', 'Erickson', 'Fletcher', 'McKinney', 'Page', 'Dawson', 'Joseph', 'Marquez', 'Reeves', 'Klein', 'Espinoza', 'Baldwin', 'Moran', 'Love', 'Robbins', 'Higgins', 'Ball', 'Curtis', 'Vega', 'Glover', 'Cain', 'Fitzgerald', 'Norman', 'Mccoy', 'Jennings', 'Chandler', 'Blair', 'Lowe', 'Shannon', 'Steele', 'Casey', 'Holland', 'Briggs', 'Little', 'Mclaughlin', 'Bishop', 'Lynch', 'Owens'
]


cities_states = [
    ("Austin", "TX"), ("Dallas", "TX"), ("Houston", "TX"), ("San Antonio", "TX"),
    ("Los Angeles", "CA"), ("San Diego", "CA"), ("San Francisco", "CA"),
    ("New York", "NY"), ("Buffalo", "NY"), ("Miami", "FL"), ("Orlando", "FL"),
    ("Seattle", "WA"), ("Chicago", "IL"), ("Atlanta", "GA"), ("Phoenix", "AZ")
]

streets = [
    "Main St", "Oak Ave", "Maple Dr", "Cedar Ln", "Pine St", "Lakeview Blvd",
    "Sunset Rd", "Hillcrest Ave", "Riverside Dr", "Parkway"
]

product_categories = [
    "electronics", "clothing", "home", "sports", "books", "beauty", "toys", "grocery"
]

brands = [
    "NovaTech", "UrbanNest", "PeakPro", "BlueCart", "BrightCo", "HomeEase",
    "SwiftGear", "EcoLite", "PrimeCraft", "DailyPlus"
]

payment_methods = ["credit_card", "debit_card", "paypal", "apple_pay", "gift_card"]
order_statuses = ["placed", "processing", "shipped", "delivered", "cancelled", "returned"]
genders = ["Male", "Female", "Non-Binary", "Prefer not to say"]
countries = ["USA"]

def random_date(start_date, end_date):
    delta_days = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, delta_days))

def make_phone_number():
    return f"{random.randint(200, 999)}-{random.randint(200, 999)}-{random.randint(1000, 9999)}"

def make_zip_code():
    return f"{random.randint(10000, 99999)}"

def make_email(first_name, last_name, customer_num):
    domains = ["gmail.com", "yahoo.com", "outlook.com", "example.com"]
    return f"{first_name.lower()}.{last_name.lower()}{customer_num}@{random.choice(domains)}"

def make_street_address():
    return f"{random.randint(100, 9999)} {random.choice(streets)}"

def maybe(probability):
    return random.random() < probability


In [0]:
# customers

customers = []

today = date.today()

for i in range(1, NUM_CUSTOMERS + 1):
    customer_id = f"CUST-{i:06d}"
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)
    full_name = f"{first_name} {last_name}"

    dob = random_date(date(1950, 1, 1), date(2006, 12, 31))
    age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

    city, state = random.choice(cities_states)

    customers.append({
        "customer_id": customer_id,
        "first_name": first_name,
        "last_name": last_name,
        "full_name": full_name,
        "email": make_email(first_name, last_name, i),
        "phone_number": make_phone_number(),
        "date_of_birth": dob.isoformat(),
        "age": age,
        "gender": random.choice(genders),
        "street_address": make_street_address(),
        "city": city,
        "state": state,
        "zip_code": make_zip_code(),
        "country": "USA"
    })

In [0]:
# bad customer data


bad_customer_indices = random.sample(range(NUM_CUSTOMERS), 45)

for idx in bad_customer_indices[:8]:
    customers[idx]["email"] = random.choice([
        "bad-email",
        "missing_at_symbol.com",
        "customer@@example.com",
        "name@",
        "@domain.com"
    ])

for idx in bad_customer_indices[8:14]:
    customers[idx]["phone_number"] = random.choice([
        "123",
        "999999",
        "phone-number",
        "555-ABCD-9999",
        ""
    ])

for idx in bad_customer_indices[14:20]:
    customers[idx]["date_of_birth"] = random.choice([
        (today + timedelta(days=random.randint(1, 1000))).isoformat(),
        "1800-01-01",
        "not-a-date",
        None
    ])

for idx in bad_customer_indices[20:26]:
    customers[idx]["age"] = random.choice([-5, -1, 0, 150, 999, None])

for idx in bad_customer_indices[26:32]:
    customers[idx]["zip_code"] = random.choice([
        "ABCDE",
        "123",
        "1234567890",
        "",
        None
    ])

for idx in bad_customer_indices[32:37]:
    customers[idx]["customer_id"] = None

for idx in bad_customer_indices[37:42]:
    customers[idx]["gender"] = random.choice([
        "UnknownValue",
        "M",
        "F",
        "",
        None
    ])

for idx in bad_customer_indices[42:45]:
    customers[idx]["country"] = random.choice([
        "Atlantis",
        "",
        None
    ])

# Create a few duplicate customer IDs
duplicate_customer_source = random.choice([c for c in customers if c["customer_id"] is not None])
for idx in random.sample(range(NUM_CUSTOMERS), 3):
    customers[idx]["customer_id"] = duplicate_customer_source["customer_id"]

In [0]:
# products

products = []

for i in range(1, NUM_PRODUCTS + 1):
    category = random.choice(product_categories)
    brand = random.choice(brands)

    products.append({
        "product_id": f"PROD-{i:05d}",
        "product_name": f"{brand} {category.title()} Item {i}",
        "product_category": category,
        "brand": brand,
        "price": round(random.uniform(5.00, 800.00), 2),
        "stock_quantity": random.randint(0, 500),
        "supplier_id": f"SUP-{random.randint(1, 50):04d}"
    })



In [0]:
# bad product data

bad_product_indices = random.sample(range(NUM_PRODUCTS), 25)

for idx in bad_product_indices[:6]:
    products[idx]["price"] = random.choice([-10.00, -1.99, 0.00, None])

for idx in bad_product_indices[6:12]:
    products[idx]["stock_quantity"] = random.choice([-5, -1, None, 999999])

for idx in bad_product_indices[12:16]:
    products[idx]["product_category"] = random.choice(["unknown_category", "", None])

for idx in bad_product_indices[16:20]:
    products[idx]["product_id"] = None

for idx in bad_product_indices[20:23]:
    products[idx]["supplier_id"] = random.choice(["", None, "BAD_SUPPLIER"])

# Create duplicate product IDs
duplicate_product_source = random.choice([p for p in products if p["product_id"] is not None])
for idx in bad_product_indices[23:25]:
    products[idx]["product_id"] = duplicate_product_source["product_id"]


In [0]:
# orders

valid_customer_ids = [c["customer_id"] for c in customers if c["customer_id"] is not None]
valid_product_ids = [p["product_id"] for p in products if p["product_id"] is not None]

orders = []

for i in range(1, NUM_ORDERS + 1):
    order_id = f"ORD-{i:07d}"

    order_date = random_date(today - timedelta(days=365), today)
    ship_date = order_date + timedelta(days=random.randint(0, 7))

    delivered = random.choice([True, False])
    delivery_date = None

    if delivered:
        delivery_date = ship_date + timedelta(days=random.randint(1, 7))

    subtotal = round(random.uniform(10.00, 1200.00), 2)
    discount_amount = round(random.choice([0, 0, 0, 5, 10, 25, 50]), 2)
    tax_amount = round(subtotal * random.uniform(0.05, 0.095), 2)
    shipping_cost = round(random.choice([0, 4.99, 7.99, 12.99, 19.99]), 2)
    order_total = round(subtotal - discount_amount + tax_amount + shipping_cost, 2)

    orders.append({
        "order_id": order_id,
        "customer_id": random.choice(valid_customer_ids),
        "order_date": order_date.isoformat(),
        "ship_date": ship_date.isoformat(),
        "order_status": random.choice(order_statuses),
        "payment_method": random.choice(payment_methods),
        "order_total": order_total,
        "discount_amount": discount_amount,
        "tax_amount": tax_amount,
        "shipping_cost": shipping_cost,
        "delivered": delivered,
        "delivery_date": delivery_date.isoformat() if delivery_date else None
    })




In [0]:
# bad order data

bad_order_indices = random.sample(range(NUM_ORDERS), 130)

# Invalid emails are in customer table; orders focus on dates, amounts, status, FK, boolean logic

for idx in bad_order_indices[:15]:
    orders[idx]["customer_id"] = random.choice([
        "CUST-999999",
        "CUST-000000",
        "UNKNOWN_CUSTOMER",
        None
    ])

for idx in bad_order_indices[15:30]:
    orders[idx]["order_total"] = random.choice([
        -100.00,
        -1.00,
        0.00,
        None
    ])

for idx in bad_order_indices[30:42]:
    orders[idx]["discount_amount"] = random.choice([
        -5.00,
        9999.00,
        None
    ])

for idx in bad_order_indices[42:54]:
    orders[idx]["tax_amount"] = random.choice([
        -3.00,
        999.99,
        None
    ])

for idx in bad_order_indices[54:66]:
    orders[idx]["shipping_cost"] = random.choice([
        -9.99,
        999.99,
        None
    ])

for idx in bad_order_indices[66:78]:
    orders[idx]["order_status"] = random.choice([
        "lost_in_space",
        "unknown",
        "complete-ish",
        "",
        None
    ])

for idx in bad_order_indices[78:90]:
    orders[idx]["payment_method"] = random.choice([
        "crypto_cash",
        "barter",
        "wire_unknown",
        "",
        None
    ])

for idx in bad_order_indices[90:102]:
    orders[idx]["delivered"] = random.choice([
        "maybe",
        "yes",
        "no",
        "",
        None
    ])

# Date consistency issues: ship date before order date
for idx in bad_order_indices[102:112]:
    od = random_date(today - timedelta(days=100), today)
    orders[idx]["order_date"] = od.isoformat()
    orders[idx]["ship_date"] = (od - timedelta(days=random.randint(1, 10))).isoformat()

# Delivery date before ship date
for idx in bad_order_indices[112:120]:
    od = random_date(today - timedelta(days=100), today)
    sd = od + timedelta(days=random.randint(1, 5))
    orders[idx]["order_date"] = od.isoformat()
    orders[idx]["ship_date"] = sd.isoformat()
    orders[idx]["delivered"] = True
    orders[idx]["delivery_date"] = (sd - timedelta(days=random.randint(1, 3))).isoformat()

# Future order dates
for idx in bad_order_indices[120:125]:
    future_order_date = today + timedelta(days=random.randint(1, 365))
    orders[idx]["order_date"] = future_order_date.isoformat()
    orders[idx]["ship_date"] = (future_order_date + timedelta(days=random.randint(1, 5))).isoformat()

# Null order IDs
for idx in bad_order_indices[125:128]:
    orders[idx]["order_id"] = None

# Duplicate order IDs
duplicate_order_source = random.choice([o for o in orders if o["order_id"] is not None])
for idx in bad_order_indices[128:130]:
    orders[idx]["order_id"] = duplicate_order_source["order_id"]

In [0]:
# struct types

from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType, BooleanType)

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("street_address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("country", StringType(), True)
])

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("ship_date", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_total", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("tax_amount", DoubleType(), True),
    StructField("shipping_cost", DoubleType(), True),
    StructField("delivered", StringType(), True),
    StructField("delivery_date", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True),
    StructField("supplier_id", StringType(), True)
])

In [0]:
# spark dataframes

customers_df = spark.createDataFrame(customers, schema=customers_schema)
orders_df = spark.createDataFrame(orders, schema=orders_schema)
products_df = spark.createDataFrame(products, schema=products_schema)

In [0]:
# raw CSV files

customers_df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/customers")
orders_df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/orders")
products_df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{RAW_PATH}/products")

In [0]:
# bronze tables

customers_df.write.mode("overwrite").format("delta").saveAsTable(f"{DATABASE_NAME}.customers_bronze")
orders_df.write.mode("overwrite").format("delta").saveAsTable(f"{DATABASE_NAME}.orders_bronze")
products_df.write.mode("overwrite").format("delta").saveAsTable(f"{DATABASE_NAME}.products_bronze")

In [0]:
# validation display

print("Synthetic ecommerce data generated successfully.")
print(f"Customers: {customers_df.count()}")
print(f"Orders: {orders_df.count()}")
print(f"Products: {products_df.count()}")

display(customers_df.limit(10))
display(orders_df.limit(10))
display(products_df.limit(10))